# nb_coord_silver_arms_ingest - DEBUG (asqa_feesandchanges only)

Copy of the ARMS ingest coordinator, filtered to one entity with step-by-step output. Not for the pipeline.


## Run Common Shared Notebooks

In [ ]:
%run "/05 - shared-libraries/nb_shared_common"

In [ ]:
%run "/05 - shared-libraries/nb_shared_metadata"

In [ ]:
%run "/05 - shared-libraries/nb_shared_logging"

In [ ]:
# Pipeline parameters
storage_account_name = ""
batch_run_id = ""
run_date = ""
pipeline_id = ""
parent_pipeline_id = ""
top_level_pipeline_id = ""
pipeline_name=""
retry_count = 0


In [ ]:
# Injected values so the notebook runs standalone for debugging
batch_run_id = "run_20261102T0007"
run_date = "20261102T0007"
pipeline_id = "89460948-7c79-459c-a20b-9d360029e56a"
parent_pipeline_id = "dcca7fd5-f8a0-41a7-969f-d507236a7ce8"
top_level_pipeline_id = "34f7c918-7ce7-42a9-a4ee-b46ce96dd74f"
storage_account_name = "deintdlsaueaasqebla"
pipeline_name = "pl_coordinator_silver_arms_ingest"
retry_count = 2


## Import Libraries and setup data lake paths

In [ ]:
#import libraries
from mdw_shared import audit, metadata, transforms
from mdw_shared.audit import *
from mdw_shared.metadata import *
from mdw_shared.transforms import *
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, trim, lower, regexp_replace, max as spark_max, to_timestamp
from mdw_shared.transforms import drop_and_register_table
from delta.tables import DeltaTable

# Get spark session and delta lake paths
spark = get_spark_session()

data_lake_paths = get_datalake_paths(storage_account_name)

BRONZE_BASE_PATH = data_lake_paths["bronze"]
SILVER_BASE_PATH = data_lake_paths["silver"]
GOLD_BASE_PATH = data_lake_paths["gold"]
METADATA_BASE_PATH = data_lake_paths["metadata"]

METADATA_CONTROL_PATH = f"{METADATA_BASE_PATH}/control"
METADATA_AUDIT_PATH = f"{METADATA_BASE_PATH}/audit"
METADATA_CONFIG_PATH = f"{METADATA_BASE_PATH}/config/silver/cleanse"

STEP_NAME_TO_RUN = STEP_SILVER_ARMS_DIRECT_INGEST
PIPELINE_NAME = pipeline_name
NOTEBOOK_NAME = "nb_coord_silver_arms_ingest"

QUARANTINE_BASE_PATH = f"{SILVER_BASE_PATH}/quarantine"


## Debug: single entity

In [ ]:
# =============================================================================
# DEBUG VERSION - processes ONE entity only and prints everything.
# Audit logging, quarantine writes and watermark updates are OFF by default.
# =============================================================================
import traceback
import uuid
from collections import Counter
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable
from notebookutils import mssparkutils

DEBUG_ENTITY = "asqa_feesandchanges"   # lowercase target entity name
WRITE_ENABLED = True        # False = run every step but skip the Delta write
UPDATE_WATERMARK = False    # True = move the watermark after a successful write
FORCE_FULL_LOAD = False     # True = ignore the watermark and read everything
SHOW_ROWS = 10


def dbg(title, msg=""):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[DBG {ts}] {title}: {msg}")


def banner(title):
    print("")
    print("=" * 100)
    print(title)
    print("=" * 100)


def dump_cfg(cfg):
    if hasattr(cfg, "asDict"):
        return cfg.asDict()
    if hasattr(cfg, "_asdict"):
        return cfg._asdict()
    try:
        return vars(cfg)
    except TypeError:
        return str(cfg)


def search_control_tables(term):
    """Scan every Delta table under the control path for rows mentioning term."""
    for item in mssparkutils.fs.ls(METADATA_CONTROL_PATH):
        try:
            d = spark.read.format("delta").load(item.path)
            str_cols = [f.name for f in d.schema.fields if isinstance(f.dataType, StringType)]
            if not str_cols:
                continue
            cond = None
            for c in str_cols:
                expr = F.lower(F.col(c)).contains(term)
                cond = expr if cond is None else (cond | expr)
            hits = d.filter(cond)
            n = hits.count()
            dbg("control table", f"{item.name} hits={n}")
            if n > 0:
                hits.show(50, truncate=False)
        except Exception as ex:
            dbg("control table skipped", f"{item.name} -> {str(ex)[:150]}")


def show_delta_info(path, label):
    try:
        dt = DeltaTable.forPath(spark, path)
        dbg(f"{label} DESCRIBE DETAIL", "")
        dt.detail().select("location", "numFiles", "sizeInBytes", "partitionColumns").show(truncate=False)
        dbg(f"{label} last 5 history entries", "")
        dt.history(5).select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)
    except Exception as ex:
        dbg(f"{label} delta info FAILED", str(ex)[:300])


# -----------------------------------------------------------------------------
banner("STEP 0: parameters and paths")
dbg("storage_account_name", storage_account_name)
dbg("batch_run_id", batch_run_id)
dbg("STEP_NAME_TO_RUN", STEP_NAME_TO_RUN)
dbg("ARMS_SOURCE_SYSTEM", ARMS_SOURCE_SYSTEM)
dbg("METADATA_CONTROL_PATH", METADATA_CONTROL_PATH)
dbg("METADATA_CONFIG_PATH", METADATA_CONFIG_PATH)
dbg("SILVER_BASE_PATH", SILVER_BASE_PATH)
dbg("DEBUG_ENTITY", DEBUG_ENTITY)
dbg("flags", f"WRITE_ENABLED={WRITE_ENABLED} UPDATE_WATERMARK={UPDATE_WATERMARK} FORCE_FULL_LOAD={FORCE_FULL_LOAD}")

# -----------------------------------------------------------------------------
banner("STEP 1: transform configs from metadata (this decides what the loop sees)")
transform_configs = get_transform_config(METADATA_CONTROL_PATH, STEP_NAME_TO_RUN)
if not transform_configs:
    raise Exception(f"get_transform_config returned nothing for step {STEP_NAME_TO_RUN}")

all_names = sorted([str(c.target_entity_name).lower() for c in transform_configs])
dbg("total transform configs returned", len(all_names))
dupes = {k: v for k, v in Counter(all_names).items() if v > 1}
dbg("entities returned MORE THAN ONCE (duplicate metadata rows)", dupes)

matches = [
    c for c in transform_configs
    if str(c.target_entity_name).lower() == DEBUG_ENTITY
    or str(c.source_entity_name).lower() == DEBUG_ENTITY
]
dbg("configs matching debug entity", len(matches))

if not matches:
    banner("ROOT CAUSE: no transform config row for " + DEBUG_ENTITY)
    dbg("near matches (contain 'fee' or 'change')", [n for n in all_names if "fee" in n or "change" in n])
    dbg("ALL entity names returned by get_transform_config", all_names)
    dbg("searching every control table for 'feesandchanges'", "")
    search_control_tables("feesandchanges")
    raise Exception(
        f"{DEBUG_ENTITY} is not returned by get_transform_config for step {STEP_NAME_TO_RUN}. "
        f"The production loop never sees it, so it is skipped silently. "
        f"Check the control table row exists and is active/enabled for this step."
    )

for m in matches:
    dbg("matched transform config", dump_cfg(m))

# -----------------------------------------------------------------------------
banner("STEP 2: ingestion configs (watermark settings)")
ingestion_configs = get_silver_ingestion_config(METADATA_CONTROL_PATH, ARMS_SOURCE_SYSTEM)
dbg("total ingestion configs", len(ingestion_configs) if ingestion_configs else ingestion_configs)
ing_matches = [
    i for i in (ingestion_configs or [])
    if str(i.target_name).lower() == DEBUG_ENTITY
    or str(i.source_object_name).lower() == DEBUG_ENTITY
]
dbg("ingestion configs matching debug entity", len(ing_matches))
for i in ing_matches:
    dbg("matched ingestion config", dump_cfg(i))
if not ing_matches:
    dbg("WARNING", "no ingestion config found - prod code would fall back to a FULL load with no watermark")

# -----------------------------------------------------------------------------
for cfg in matches:
    entity_name = cfg.target_entity_name.lower()
    source_entity = cfg.source_entity_name
    target_path = cfg.target_path.lower()
    source_path = cfg.source_path.lower()
    act_id = str(uuid.uuid4())

    banner(f"STEP 3: processing {entity_name}  (source_entity={source_entity})")
    dbg("source_path", source_path)
    dbg("target_path", target_path)
    dbg("source path exists", mssparkutils.fs.exists(source_path))
    dbg("target path exists", mssparkutils.fs.exists(target_path))

    # Exact same lookup as prod, so a mismatch shows up here
    ingest_cfg = next(
        (
            icfg for icfg in (ingestion_configs or [])
            if icfg.source_object_name == source_entity
            and icfg.target_name == entity_name
        ),
        None
    )
    dbg("ingest_cfg matched with prod's exact-match rule", ingest_cfg is not None)

    ingestion_mapping_id = ingest_cfg.ingestion_mapping_id if ingest_cfg else None
    ingestion_mode = (ingest_cfg.ingestion_mode if ingest_cfg else "full").lower()
    watermark_column = ingest_cfg.watermark_column if ingest_cfg else None
    last_success_watermark_value = ingest_cfg.last_success_watermark_value if ingest_cfg else None
    if FORCE_FULL_LOAD:
        ingestion_mode = "full"
    dbg("ingestion_mapping_id", ingestion_mapping_id)
    dbg("ingestion_mode", ingestion_mode)
    dbg("watermark_column", watermark_column)
    dbg("last_success_watermark_value", last_success_watermark_value)

    try:
        # ---- 3a. read source ----
        banner("3a. Read source")
        df = spark.read.format("delta").load(source_path)
        dbg("source columns", df.columns)
        df.printSchema()
        src_total = df.count()
        dbg("SOURCE TOTAL ROWS (before any filter)", src_total)
        if src_total == 0:
            dbg("ROOT CAUSE?", "source delta table at source_path is EMPTY. The SQL view you queried may point somewhere else.")
        show_delta_info(source_path, "SOURCE")

        if watermark_column and watermark_column in df.columns:
            df.agg(
                F.min(F.col(watermark_column)).alias("min_wm"),
                F.max(F.col(watermark_column)).alias("max_wm")
            ).show(truncate=False)
        elif watermark_column:
            dbg("WARNING", f"watermark column {watermark_column} NOT in source columns")

        # ---- 3b. incremental filter ----
        banner("3b. Incremental filter")
        if ingestion_mode == "incremental":
            if watermark_column not in df.columns:
                dbg("filter", "watermark column missing in source -> full snapshot")
            elif not last_success_watermark_value:
                dbg("filter", "no previous watermark -> all records")
            else:
                wm_type = str(df.schema[watermark_column].dataType)
                wm_value = str(last_success_watermark_value)
                dbg("filter", f"{watermark_column} ({wm_type}) > {wm_value}")
                incremental_filter = F.col(watermark_column) > F.lit(wm_value)
                if "TimestampType" in wm_type or "DateType" in wm_type:
                    incremental_filter = F.col(watermark_column) > F.to_timestamp(F.lit(wm_value))
                df = df.filter(incremental_filter)
        else:
            dbg("filter", "full mode - no filter")

        rows_read = df.count()
        dbg("ROWS AFTER INCREMENTAL FILTER (rows_read)", rows_read)
        watermark_df = df

        if rows_read == 0:
            banner("STOPPING: 0 rows after incremental filter")
            dbg("WHY NOTHING IS WRITTEN",
                "prod logs 'no incremental changes', marks the activity succeeded and moves on. No error is raised.")
            dbg("compare", f"source total={src_total}, last_success_watermark_value={last_success_watermark_value}")
            continue

        # ---- 3c. rules ----
        banner("3c. Validation rules / column mappings")
        rules = load_validation_rules(f"{ARMS_SOURCE_SYSTEM}/{entity_name}", metadata_config_path=METADATA_CONFIG_PATH)
        dbg("rules loaded", bool(rules))
        if not rules:
            raise Exception(f"No or malformed rules JSON for {ARMS_SOURCE_SYSTEM}/{entity_name} under {METADATA_CONFIG_PATH}")
        dbg("rules keys", list(rules.keys()))
        dbg("primary_key", rules.get("primary_key"))
        dbg("validations", rules.get("validations"))
        dbg("column_mappings", rules.get("column_mappings"))

        entity_quarantined = 0
        rules_applied = 0

        if "column_mappings" in rules:
            before_cols = df.columns
            df = apply_column_mappings(df, rules["column_mappings"])
            dbg("columns BEFORE mappings", before_cols)
            dbg("columns AFTER mappings", df.columns)
            dbg("rows after mappings", df.count())

        if "validations" in rules:
            rules_applied = len(rules["validations"])
            df, invalid_df = apply_validations(df, rules["validations"])
            valid_n = df.count()
            invalid_n = invalid_df.count()
            entity_quarantined = invalid_n
            dbg("VALID rows after validations", valid_n)
            dbg("INVALID rows (would be quarantined)", invalid_n)
            if invalid_n > 0:
                invalid_df.show(SHOW_ROWS, truncate=False)
            if valid_n == 0:
                dbg("ROOT CAUSE?", "validations rejected every row - nothing left to write")

        # ---- 3d. text standardise ----
        banner("3d. Text standardise")
        string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType"]
        dbg("string cols found by prod's check (str(dataType)=='StringType')", len(string_cols))
        for c in string_cols:
            df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
        dbg("rows after text standardise", df.count())

        # ---- 3e. dedupe ----
        banner("3e. Deduplicate")
        if "primary_key" in rules:
            pk = rules["primary_key"]
            missing_pk = [k for k in pk if k not in df.columns]
            dbg("primary key columns missing from df", missing_pk)
            order_by_col = try_get_col_name(df, ["modifiedon", "modified_on", "SinkModifiedOn", "sink_modified_on"])
            dbg("order_by_col", order_by_col)
            before = df.count()
            if order_by_col is not None:
                df = deduplicate(df, pk, order_by_col)
            after = df.count()
            dbg("rows before/after dedupe", f"{before} -> {after}")
            assert_no_duplicates(df, pk, source_entity)
        else:
            dbg("dedupe", "no primary_key in rules - skipped")

        # ---- 3f. target schema compare + cast ----
        banner("3f. Target schema vs df schema")
        target_df = spark.read.format("delta").load(target_path)
        target_schema = {f.name.lower(): f.dataType for f in target_df.schema.fields}
        dbg("TARGET ROWS BEFORE WRITE (all batches)", target_df.count())
        show_delta_info(target_path, "TARGET")

        df_cols_lower = {c.lower() for c in df.columns}
        dbg("columns in df but NOT in target (mergeSchema will add)", [c for c in df.columns if c.lower() not in target_schema])
        dbg("columns in target but NOT in df (will be null)", [c for c in target_schema if c not in df_cols_lower])

        for field in df.schema.fields:
            existing_type = target_schema.get(field.name.lower())
            if existing_type and existing_type != field.dataType:
                dbg("CAST", f"{field.name}: {field.dataType} -> {existing_type}")
                df = df.withColumn(field.name, F.col(field.name).cast(existing_type))
        dbg("rows after casts", df.count())

        # ---- 3g. metadata columns ----
        banner("3g. Silver metadata columns")
        df = derive_change_type(df)
        df = add_silver_transaction_metadata(df, batch_run_id, act_id, source_system=ARMS_SOURCE_SYSTEM)
        df = df.withColumn("mdw_source_entity", F.lit(source_entity))
        rows_written = df.count()
        dbg("ROWS TO WRITE", rows_written)
        df.printSchema()
        df.show(SHOW_ROWS, truncate=False)

        # ---- 3h. write ----
        banner("3h. Write")
        if not WRITE_ENABLED:
            dbg("WRITE_ENABLED is False", "skipping delete + write")
        elif rows_written == 0:
            dbg("WRITE SKIPPED", "0 rows to write")
        else:
            delete_batch_data(spark, target_path, batch_run_id)
            dbg("delete_batch_data", "done")
            df.write.format("delta").mode("append").option("mergeSchema", "true").save(target_path)
            dbg("append", "done")

            after_df = spark.read.format("delta").load(target_path)
            dbg("TARGET TOTAL ROWS AFTER WRITE", after_df.count())
            dbg("TARGET ROWS FOR THIS batch_run_id", after_df.filter(F.col("mdw_batch_run_id") == batch_run_id).count())
            show_delta_info(target_path, "TARGET AFTER")
            dbg("NOTE", "if these counts are > 0 but silver2_db view is empty, the view points at a different path than target_path above")

            if UPDATE_WATERMARK and ingestion_mapping_id is not None and watermark_column:
                max_wm = None
                if watermark_column in watermark_df.columns:
                    max_wm = watermark_df.agg(F.max(F.col(watermark_column))).collect()[0][0]
                update_watermark(METADATA_CONTROL_PATH, ingestion_mapping_id, max_wm)
                dbg("watermark updated to", max_wm)
            else:
                dbg("watermark", "NOT updated (UPDATE_WATERMARK=False)")

    except Exception:
        banner(f"FAILED while processing {entity_name}")
        traceback.print_exc()
        raise

banner("DEBUG RUN COMPLETE")
